In [1]:
# 문제 6: SMS 스팸 이진 분류 (macro F1). 근거: RESEARCH.md, PLAN.md
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, TunedThresholdClassifierCV
from sklearn.metrics import f1_score, make_scorer, confusion_matrix

T0 = time.time()
DATA = "data"
OUT = "submission.csv"
macro_f1 = make_scorer(f1_score, average="macro")

In [2]:
train = pd.read_csv(f"{DATA}/train.csv", encoding="utf-8-sig")
test = pd.read_csv(f"{DATA}/test.csv", encoding="utf-8-sig")
sample = pd.read_csv(f"{DATA}/sample_submission.csv", encoding="utf-8-sig")
X, y = train["message"], train["label"].values
print(train.shape, test.shape, sample.shape)
print("결측:", train.isna().sum().sum(), test.isna().sum().sum())
print("라벨 분포:", train["label"].value_counts().to_dict(), f"스팸 비율 {y.mean():.3f}")

(4000, 3) (1000, 2) (1000, 2)
결측: 0 0
라벨 분포: {0: 3502, 1: 498} 스팸 비율 0.124


In [3]:
def char_vec():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=2)

def word_vec():
    return TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)

def tuned(base):
    return TunedThresholdClassifierCV(base, scoring=macro_f1, thresholds=41,
                                      cv=StratifiedKFold(3, shuffle=True, random_state=7))

A = make_pipeline(char_vec(), LinearSVC(C=1, class_weight="balanced"))
CANDS = {
    "A": A,
    "Cthr": tuned(clone(A)),
    "Ethr": tuned(make_pipeline(make_union(word_vec(), char_vec()), LinearSVC(C=1, class_weight="balanced"))),
}

In [4]:
# 안전 제출본: A를 전체 train에 학습해 먼저 저장
def save(model):
    sub = pd.DataFrame({"message_id": test["message_id"], "label": model.predict(test["message"]).astype(int)})
    sub.to_csv(OUT, index=False, encoding="utf-8-sig")
    return sub

save(clone(A).fit(X, y))
print("안전 제출본 저장")

안전 제출본 저장


In [5]:
# 15 fold 짝지은 비교 (RESEARCH와 다른 seed 2026~2028)
scores = {k: [] for k in CANDS}
oof = {k: np.zeros((3, len(y)), dtype=int) for k in CANDS}
for r in range(3):
    for a, b in StratifiedKFold(5, shuffle=True, random_state=2026 + r).split(X, y):
        for k, p in CANDS.items():
            pred = clone(p).fit(X.iloc[a], y[a]).predict(X.iloc[b])
            oof[k][r, b] = pred
            scores[k].append(f1_score(y[b], pred, average="macro"))

base = np.array(scores["A"])
rows = []
for k, v in scores.items():
    v = np.array(v); d = v - base
    pooled = np.mean([f1_score(y, oof[k][r], average="macro") for r in range(3)])
    rows.append({"model": k, "fold_mean": v.mean(), "fold_std": v.std(), "pooled_oof": pooled,
                 "diff_vs_A": d.mean(), "win": int((d > 0).sum()), "lose": int((d < 0).sum())})
res = pd.DataFrame(rows).set_index("model")
display(res.round(4))
for k in CANDS:
    print(k, "OOF 혼동 [TN FP FN TP] (seed 2026):", confusion_matrix(y, oof[k][0]).ravel())
fn = train.loc[(y == 1) & (oof["A"][0] == 0), "message"]
print("\nA가 놓친 스팸 예시:"); print("\n".join("- " + m[:110] for m in fn.head(6)))

,fold_mean,fold_std,pooled_oof,diff_vs_A,win,lose
model,,,,,,
A,0.9818,0.0069,0.9818,0.0000,0,0
Cthr,0.9832,0.0065,0.9833,0.0015,8,3
Ethr,0.9824,0.0059,0.9824,0.0006,8,4


A OOF 혼동 [TN FP FN TP] (seed 2026): [3498    4   30  468]
Cthr OOF 혼동 [TN FP FN TP] (seed 2026): [3498    4   26  472]
Ethr OOF 혼동 [TN FP FN TP] (seed 2026): [3493    9   22  476]

A가 놓친 스팸 예시:
- Are you unique enough? Find out from 30th August. www.areyouunique.co.uk
- (Bank of Granite issues Strong-Buy) EXPLOSIVE PICK FOR OUR MEMBERS *****UP OVER 300% *********** Nasdaq Symbol
- Money i have won wining number 946 wot do i do next
- How come it takes so little time for a child who is afraid of the dark to become a teenager who wants to stay 
- In The Simpsons Movie released in July 2007 name the band that died at the start of the film? A-Green Day, B-B
- Will u meet ur dream partner soon? Is ur career off 2 a flyng start? 2 find out free, txt HORO followed by ur 


In [6]:
# PLAN §4 사전 채택 규칙: diff > 0 이고 win > lose + 3
passed = [k for k in ["Cthr", "Ethr"] if res.loc[k, "diff_vs_A"] > 0 and res.loc[k, "win"] > res.loc[k, "lose"] + 3]
if not passed:
    choice = "A"
elif len(passed) == 2:
    dc, de = res.loc["Cthr", "diff_vs_A"], res.loc["Ethr", "diff_vs_A"]
    choice = "Ethr" if de - dc > 0.001 else "Cthr"
else:
    choice = passed[0]
print("규칙 통과:", passed, "→ 선택:", choice)

final = clone(CANDS[choice]).fit(X, y)  # 임계값은 전체 train 내부 3-fold로 다시 선택
if choice != "A":
    print("선택된 임계값:", round(float(final.best_threshold_), 4))
sub = save(final)

규칙 통과: ['Cthr', 'Ethr'] → 선택: Cthr


선택된 임계값: -0.0781


In [7]:
# 형식 검사
chk = pd.read_csv(OUT, encoding="utf-8-sig")
assert len(chk) == 1000
assert list(chk.columns) == ["message_id", "label"]
assert (chk["message_id"] == sample["message_id"]).all()
assert set(chk["label"].unique()) <= {0, 1}
assert chk.isna().sum().sum() == 0
rate = chk["label"].mean()
print(f"형식 검사 통과. 예측 스팸 비율 {rate:.3f} (train {y.mean():.3f})")
if abs(rate - y.mean()) > 0.05:
    print("경고: 예측 스팸 비율이 train과 크게 다름")
print(f"전체 실행 시간 {time.time() - T0:.1f}s")

형식 검사 통과. 예측 스팸 비율 0.118 (train 0.124)
전체 실행 시간 65.3s
